# HMM with Rate of Change and Yang-Zhang Volatility

Runs a direct walk-forward HMM with price momentum and range-based volatility.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
HMM Regime Detection Trading Strategy - ROC63 & Yang-Zhang Volatility
======================================================================

CRITICAL: This implementation uses STRICT walk-forward validation to eliminate
ALL look-ahead bias. HMM training is done in expanding windows with regime
mapping frozen for each prediction period.

Key Differences from Previous Model:
------------------------------------
- NO Ridge regression (removed completely)
- Features: ROC63, Yang-Zhang Volatility Percentiles
- Direct HMM training on standardized features
- Simpler, more interpretable model

Features Used:
--------------
1. ROC63: Rate of Change over 63 trading days (1 quarter)
   - Captures medium-term momentum
   - Formula: (Price[t] - Price[t-63]) / Price[t-63]

2. Yang-Zhang Volatility:
   - More efficient than Garman-Klass or simple historical volatility
   - Uses Open, High, Low, Close prices
   - Formula combines overnight, intraday, and Rogers-Satchell components
   - Percentile rank over rolling window for regime detection

Trading Logic:
--------------
- Signal generated at Close[t] based on regime prediction
- Execution at Open[t+1]
- Portfolio valued at Open (not Close) on execution days
- Returns calculated as Open[t+1]/Open[t] - 1 (Open-to-Open)
- During holding: Close-to-Close returns
- Benchmark: Close-to-Close returns (no executions)
- No look-ahead bias (strictly T-1 for T predictions)

Author: Production Trading System
Date: 2025
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, Tuple, Optional, List
from sklearn.preprocessing import StandardScaler
from hmmlearn import hmm
from dataclasses import dataclass

warnings.filterwarnings("ignore")


class StrategyConfig:
    """Centralized configuration for all strategy parameters."""

    START_DATE = "2019-01-01"
    END_DATE = "2024-12-31"
    INITIAL_CAPITAL = 100000.0
    COMMISSION_RATE = 0.001
    MAIN_TICKER = "SPY"
    BENCHMARK_TICKER = "URTH"
    ROC_PERIOD = 63
    YANG_ZHANG_WINDOW = 20
    PERCENTILE_WINDOW = 252
    HMM_N_STATES = 3
    HMM_N_ITER = 100
    HMM_RANDOM_STATE = 42
    MIN_TRAINING_DAYS = 252
    REFIT_FREQUENCY = 63


class DataFetcher:
    """
    Handles all data acquisition from Yahoo Finance.

    IMPORTANT: No bfill or ffill - we work only with available data.
    Missing data is handled by dropping NaN rows after feature calculation.
    """

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Fetch historical OHLCV data from Yahoo Finance.

        Parameters:
        -----------
        ticker : str
            Ticker symbol to fetch
        start_date : str
            Start date in YYYY-MM-DD format
        end_date : str
            End date in YYYY-MM-DD format

        Returns:
        --------
        pd.DataFrame with columns: Date, Open, High, Low, Close, Volume

        Notes:
        ------
        - No forward filling or backward filling
        - Returns raw data as received from API
        - Sorted by date ascending
        """
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
            if "historical" not in data:
                raise ValueError(f"No data returned for {ticker}")
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            df = df.rename(
                columns={
                    "date": "Date",
                    "open": "Open",
                    "high": "High",
                    "low": "Low",
                    "close": "Close",
                    "volume": "Volume",
                }
            )
            df = df[["Date", "Open", "High", "Low", "Close", "Volume"]]
            print(
                f"✓ Fetched {len(df)} records for {ticker} ({df['Date'].min().date()} to {df['Date'].max().date()})"
            )
            return df
        except Exception as e:
            print(f"✗ Error fetching data for {ticker}: {str(e)}")
            raise


class FeatureEngine:
    """
    Calculates ROC63 and Yang-Zhang volatility features.
    All calculations avoid look-ahead bias by using only past data.
    """

    @staticmethod
    def calculate_roc(close: pd.Series, period: int) -> pd.Series:
        """
        Calculate Rate of Change (ROC) over specified period.

        Formula: (Close[t] - Close[t-period]) / Close[t-period]

        Parameters:
        -----------
        close : pd.Series
            Close prices
        period : int
            Number of periods to look back

        Returns:
        --------
        pd.Series with ROC values

        Notes:
        ------
        - Measures momentum over the period
        - Positive values indicate upward momentum
        - Negative values indicate downward momentum
        - First 'period' values will be NaN (no look-ahead)
        """
        return close.pct_change(periods=period)

    @staticmethod
    def calculate_yang_zhang_volatility(df: pd.DataFrame, window: int) -> pd.Series:
        """
        Calculate Yang-Zhang volatility estimator.

        Yang-Zhang is a volatility estimator that combines:
        1. Overnight volatility (open jump from previous close)
        2. Intraday volatility (open to close)
        3. Rogers-Satchell component (high-low range)

        Formula:
        --------
        σ²_YZ = σ²_o + k·σ²_c + (1-k)·σ²_rs

        Where:
        - σ²_o: overnight variance = E[(ln(O[t]/C[t-1]))²]
        - σ²_c: close-to-close variance = E[(ln(C[t]/C[t-1]))²]
        - σ²_rs: Rogers-Satchell variance = E[ln(H/C)·ln(H/O) + ln(L/C)·ln(L/O)]
        - k: weight constant = 0.34 / (1.34 + (n+1)/(n-1))

        Parameters:
        -----------
        df : pd.DataFrame
            Must contain Open, High, Low, Close columns
        window : int
            Rolling window for variance calculation

        Returns:
        --------
        pd.Series with annualized Yang-Zhang volatility

        Notes:
        ------
        - More efficient than Garman-Klass or simple historical vol
        - Accounts for overnight gaps (open != previous close)
        - Annualized by multiplying by sqrt(252)
        - First 'window' values will be NaN (no look-ahead)
        """
        log_o = np.log(df["Open"] / df["Close"].shift(1))
        log_c = np.log(df["Close"] / df["Close"].shift(1))
        log_h_c = np.log(df["High"] / df["Close"])
        log_h_o = np.log(df["High"] / df["Open"])
        log_l_c = np.log(df["Low"] / df["Close"])
        log_l_o = np.log(df["Low"] / df["Open"])
        rs = log_h_c * log_h_o + log_l_c * log_l_o
        n = window
        k = 0.34 / (1.34 + (n + 1) / (n - 1))
        var_o = log_o.rolling(window=window, min_periods=window).var()
        var_c = log_c.rolling(window=window, min_periods=window).var()
        var_rs = rs.rolling(window=window, min_periods=window).mean()
        variance_yz = var_o + k * var_c + (1 - k) * var_rs
        volatility_yz = np.sqrt(variance_yz) * np.sqrt(252)
        return volatility_yz

    @staticmethod
    def calculate_percentile_rank(series: pd.Series, window: int) -> pd.Series:
        """
        Calculate rolling percentile rank of a series.

        Returns value between 0 and 1 indicating where current value
        ranks within the rolling window.

        Parameters:
        -----------
        series : pd.Series
            Input series (e.g., volatility)
        window : int
            Rolling window for percentile calculation

        Returns:
        --------
        pd.Series with percentile ranks (0 to 1)

        Notes:
        ------
        - 0.0 means current value is minimum in window
        - 0.5 means current value is median in window
        - 1.0 means current value is maximum in window
        - Useful for regime detection (high vol percentile = stressed regime)
        - First 'window' values will be NaN (no look-ahead)
        """

        def percentile_rank(x):
            """Calculate percentile rank of last value in rolling window."""
            if len(x) < window:
                return np.nan
            return (x < x.iloc[-1]).sum() / len(x)

        return series.rolling(window=window, min_periods=window).apply(percentile_rank, raw=False)

    @staticmethod
    def prepare_features(df_main: pd.DataFrame, config: StrategyConfig) -> pd.DataFrame:
        """
        Prepare all features with proper lagging to avoid look-ahead bias.

        Feature Pipeline:
        -----------------
        1. Calculate ROC63 (momentum over 1 quarter)
        2. Calculate Yang-Zhang volatility (20-day window)
        3. Calculate volatility percentile rank (252-day window)
        4. Lag all features by 1 day (use T-1 data to predict T)

        Parameters:
        -----------
        df_main : pd.DataFrame
            Main OHLCV dataframe
        config : StrategyConfig
            Configuration object with parameters

        Returns:
        --------
        pd.DataFrame with original columns plus lagged features

        Notes:
        ------
        - All features are lagged by 1 day to prevent look-ahead
        - NaN values are NOT filled - will be dropped later
        - Features represent information available at T-1 to predict T
        """
        df = df_main.copy()
        df["Returns_OO"] = df["Open"].pct_change()
        df["Returns_CC"] = df["Close"].pct_change()
        print("\nCalculating features...")
        df["ROC63"] = FeatureEngine.calculate_roc(df["Close"], config.ROC_PERIOD)
        print(f"  ✓ ROC63 calculated (period={config.ROC_PERIOD} days)")
        df["YZ_Vol"] = FeatureEngine.calculate_yang_zhang_volatility(df, config.YANG_ZHANG_WINDOW)
        print(f"  ✓ Yang-Zhang volatility calculated (window={config.YANG_ZHANG_WINDOW} days)")
        df["YZ_Vol_Pct"] = FeatureEngine.calculate_percentile_rank(
            df["YZ_Vol"], config.PERCENTILE_WINDOW
        )
        print(f"  ✓ Yang-Zhang percentile calculated (window={config.PERCENTILE_WINDOW} days)")
        feature_cols = ["ROC63", "YZ_Vol", "YZ_Vol_Pct"]
        for col in feature_cols:
            df[f"{col}_lag"] = df[col].shift(1)
        print("  ✓ All features lagged by 1 day (T-1 for T prediction)")
        print("\nFeature summary (after lagging):")
        lagged_cols = [f"{col}_lag" for col in feature_cols]
        for col in lagged_cols:
            valid_count = df[col].notna().sum()
            print(f"  {col}: {valid_count} valid observations")
        return df


@dataclass
class RegimeMapping:
    """
    Stores regime mapping from training period.

    This mapping is created once per training window and then FROZEN
    for all predictions in that window. This prevents look-ahead bias.
    """

    state_to_regime: Dict[int, str]
    mean_returns: Dict[int, float]
    training_end_date: pd.Timestamp


class WalkForwardHMMRegimeDetector:
    """
    HMM Regime Detector with STRICT walk-forward validation.

    This implementation is SIMPLER than the Ridge version:
    - Direct standardization of features (no Ridge transformation)
    - HMM trained on standardized features
    - Regime mapping based on forward returns

    Walk-Forward Process:
    ---------------------
    For each prediction window:
    1. Fit StandardScaler on training data
    2. Transform training data
    3. Fit HMM on transformed training data
    4. Create regime mapping using ONLY training forward returns
    5. Transform next observation with frozen scaler
    6. Predict regime for next observation with frozen HMM/mapping
    7. Move window forward and repeat

    NO FUTURE DATA is ever used:
    - Feature values are from T-1
    - HMM trained on past data only
    - Regime mapping uses past forward returns only
    - Each prediction is truly out-of-sample
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.feature_cols = ["ROC63_lag", "YZ_Vol_lag", "YZ_Vol_Pct_lag"]

    def walk_forward_predict(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, List[Dict]]:
        """
        Perform walk-forward prediction with NO look-ahead bias.

        Process for each window:
        1. Fit StandardScaler on training data (expanding window)
        2. Transform training data to zero mean, unit variance
        3. Fit HMM on transformed training data
        4. Create regime mapping using ONLY training forward returns
        5. Transform current observation with FROZEN scaler
        6. Predict regime for current observation with FROZEN HMM and mapping
        7. Move to next day and repeat (refit every REFIT_FREQUENCY days)

        Parameters:
        -----------
        df : pd.DataFrame
            Dataframe with lagged features

        Returns:
        --------
        Tuple of:
        - df_clean: DataFrame with regime predictions
        - folds_executed: List of fold information for validation

        Notes:
        ------
        - Expanding window: training set grows over time
        - Scaler, HMM, and mapping are frozen between refits
        - Refit frequency controls model adaptiveness vs. stability
        """
        print("\n" + "=" * 100)
        print("WALK-FORWARD REGIME DETECTION (NO RIDGE - DIRECT HMM)")
        print("=" * 100)
        df_clean = df.dropna(subset=self.feature_cols).copy().reset_index(drop=True)
        print(f"\nTotal observations after cleaning: {len(df_clean)}")
        print(f"Date range: {df_clean['Date'].min().date()} to {df_clean['Date'].max().date()}")
        df_clean["Regime_State"] = np.nan
        df_clean["Regime"] = None
        min_train = self.config.MIN_TRAINING_DAYS
        refit_freq = self.config.REFIT_FREQUENCY
        folds_executed = []
        print(f"\nWalk-forward configuration:")
        print(f"  Minimum training days: {min_train}")
        print(f"  Refit frequency: {refit_freq} days")
        print(f"  Total periods to predict: {len(df_clean) - min_train}")
        print(f"  Expected number of refits: {(len(df_clean) - min_train) // refit_freq + 1}\n")
        last_refit_idx = min_train
        scaler = None
        hmm_model = None
        regime_mapping = None
        fold_num = 0
        for i in range(min_train, len(df_clean)):
            days_since_refit = i - last_refit_idx
            if scaler is None or days_since_refit >= refit_freq:
                fold_num += 1
                train_start = 0
                train_end = i
                print(f"\n{'=' * 100}")
                print(f"FOLD {fold_num}: Refitting models")
                print(f"{'=' * 100}")
                print(f"Training window: index {train_start} to {train_end - 1}")
                print(
                    f"Training dates: {df_clean.loc[train_start, 'Date'].date()} to {df_clean.loc[train_end - 1, 'Date'].date()}"
                )
                print(f"Training size: {train_end - train_start} days")
                X_train_raw = df_clean.loc[train_start : train_end - 1, self.feature_cols].values
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_raw)
                print(f"✓ StandardScaler fitted")
                print(f"  Feature means: {scaler.mean_}")
                print(f"  Feature stds: {scaler.scale_}")
                hmm_model = hmm.GaussianHMM(
                    n_components=self.config.HMM_N_STATES,
                    covariance_type="full",
                    n_iter=self.config.HMM_N_ITER,
                    random_state=self.config.HMM_RANDOM_STATE,
                )
                hmm_model.fit(X_train_scaled)
                print(f"✓ HMM fitted (converged: {hmm_model.monitor_.converged})")
                states_train = hmm_model.predict(X_train_scaled)
                returns_forward_train = (
                    df_clean.loc[train_start : train_end - 1, "Returns_OO"].shift(-1).values
                )
                state_returns = {}
                state_counts = {}
                for state in range(self.config.HMM_N_STATES):
                    mask = states_train == state
                    mask[-1] = False
                    count = mask.sum()
                    state_counts[state] = count
                    if count > 0:
                        state_returns[state] = returns_forward_train[mask].mean()
                    else:
                        state_returns[state] = 0
                sorted_states = sorted(state_returns.items(), key=lambda x: x[1])
                state_to_regime = {
                    sorted_states[0][0]: "Bearish",
                    sorted_states[1][0]: "Neutral",
                    sorted_states[2][0]: "Bullish",
                }
                regime_mapping = RegimeMapping(
                    state_to_regime=state_to_regime,
                    mean_returns=state_returns,
                    training_end_date=df_clean.loc[train_end - 1, "Date"],
                )
                print(f"✓ Regime mapping created (using ONLY training forward returns):")
                for state, regime in state_to_regime.items():
                    print(
                        f"  State {state} → {regime}: {state_counts[state]} obs, avg forward return: {state_returns[state]:.6f}"
                    )
                last_refit_idx = i
                folds_executed.append(
                    {
                        "fold": fold_num,
                        "train_start": train_start,
                        "train_end": train_end,
                        "train_start_date": df_clean.loc[train_start, "Date"],
                        "train_end_date": df_clean.loc[train_end - 1, "Date"],
                        "regime_mapping": regime_mapping.state_to_regime.copy(),
                        "mean_returns": regime_mapping.mean_returns.copy(),
                    }
                )
            X_current_raw = df_clean.loc[i:i, self.feature_cols].values
            X_current_scaled = scaler.transform(X_current_raw)
            state_pred = hmm_model.predict(X_current_scaled)[0]
            regime_pred = regime_mapping.state_to_regime[state_pred]
            df_clean.loc[i, "Regime_State"] = state_pred
            df_clean.loc[i, "Regime"] = regime_pred
        print(f"\n{'=' * 100}")
        print(f"WALK-FORWARD COMPLETE")
        print(f"{'=' * 100}")
        print(f"Total folds executed: {len(folds_executed)}")
        print(f"Predictions generated: {df_clean['Regime'].notna().sum()}")
        regime_counts = df_clean["Regime"].value_counts()
        print(f"\nFinal regime distribution:")
        for regime in ["Bearish", "Neutral", "Bullish"]:
            count = regime_counts.get(regime, 0)
            pct = count / len(df_clean) * 100 if len(df_clean) > 0 else 0
            print(f"  {regime}: {count} days ({pct:.1f}%)")
        return (df_clean, folds_executed)


class BacktestEngine:
    """
    Realistic backtesting engine with proper signal-execution-return alignment.

    Key Design Decisions:
    ---------------------
    1. Signal Generation: At Close[t] based on regime prediction
    2. Execution: At Open[t+1] (realistic - can't trade at close)
    3. Portfolio Valuation:
       - On execution days (buy/sell): Value at Open[t+1]
       - During holding: Value at Close (daily marking)
    4. Returns:
       - Execution days: Open[t+1] / Open[t] - 1
       - Holding days: Close[t] / Close[t-1] - 1
    5. Commissions: Applied on both buys and sells
    """

    def __init__(self, initial_capital: float, commission_rate: float):
        self.initial_capital = initial_capital
        self.commission_rate = commission_rate
        self.trades = []

    def run_backtest(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Execute backtest with proper signal-position-return alignment.

        Trading Rules:
        --------------
        - Long when Regime == "Bullish"
        - Cash when Regime != "Bullish"

        Execution Logic:
        ----------------
        - Signal at Close[t] → Execute at Open[t+1]
        - Position[t] = Signal[t-1] (position active from execution)
        - Returns[t]:
          * If execution day: Open[t]/Open[t-1] - 1
          * If holding day: Close[t]/Close[t-1] - 1

        Parameters:
        -----------
        df : pd.DataFrame
            Dataframe with Regime predictions

        Returns:
        --------
        pd.DataFrame with backtest results including:
        - Signal: Trading signal (0=cash, 1=long)
        - Position: Actual position (0=cash, 1=long)
        - Strategy_Returns: Strategy returns
        - Portfolio_Value: Portfolio value over time
        """
        results = df.reset_index(drop=True).copy()
        results["Signal"] = 0
        results.loc[results["Regime"] == "Bullish", "Signal"] = 1
        results["Position"] = 0
        results["Cash"] = 0.0
        results["Holdings"] = 0.0
        results["Portfolio_Value"] = 0.0
        results["Strategy_Returns"] = 0.0
        results["Trades"] = ""
        cash = self.initial_capital
        shares = 0
        results.loc[0, "Cash"] = cash
        results.loc[0, "Portfolio_Value"] = cash
        print("\nExecuting backtest...")
        for i in range(1, len(results)):
            desired_position = int(results.loc[i - 1, "Signal"])
            current_position = int(results.loc[i - 1, "Position"])
            execution_happened = False
            if desired_position != current_position:
                execution_price = results.loc[i, "Open"]
                if desired_position == 1 and current_position == 0:
                    commission = cash * self.commission_rate
                    available_for_shares = cash - commission
                    shares = available_for_shares / execution_price
                    cash = 0
                    self.trades.append(
                        {
                            "Date": results.loc[i, "Date"],
                            "Type": "BUY",
                            "Price": execution_price,
                            "Shares": shares,
                            "Commission": commission,
                            "Regime": results.loc[i - 1, "Regime"],
                        }
                    )
                    results.loc[i, "Trades"] = "BUY"
                    execution_happened = True
                elif desired_position == 0 and current_position == 1:
                    gross_proceeds = shares * execution_price
                    commission = gross_proceeds * self.commission_rate
                    cash = gross_proceeds - commission
                    self.trades.append(
                        {
                            "Date": results.loc[i, "Date"],
                            "Type": "SELL",
                            "Price": execution_price,
                            "Shares": shares,
                            "Commission": commission,
                            "Regime": results.loc[i - 1, "Regime"],
                        }
                    )
                    results.loc[i, "Trades"] = "SELL"
                    shares = 0
                    execution_happened = True
            results.loc[i, "Position"] = 1 if shares > 0 else 0
            results.loc[i, "Cash"] = cash
            if execution_happened:
                results.loc[i, "Holdings"] = shares * results.loc[i, "Open"]
                results.loc[i, "Portfolio_Value"] = cash + shares * results.loc[i, "Open"]
                if results.loc[i - 1, "Portfolio_Value"] > 0:
                    results.loc[i, "Strategy_Returns"] = (
                        results.loc[i, "Portfolio_Value"] / results.loc[i - 1, "Portfolio_Value"]
                        - 1
                    )
            else:
                results.loc[i, "Holdings"] = shares * results.loc[i, "Close"]
                results.loc[i, "Portfolio_Value"] = cash + shares * results.loc[i, "Close"]
                if results.loc[i - 1, "Portfolio_Value"] > 0:
                    results.loc[i, "Strategy_Returns"] = (
                        results.loc[i, "Portfolio_Value"] / results.loc[i - 1, "Portfolio_Value"]
                        - 1
                    )
        results["Cumulative_Returns"] = (1 + results["Strategy_Returns"]).cumprod()
        print(f"✓ Backtest completed")
        print(f"  Total trades: {len(self.trades)}")
        print(f"  Buy trades: {sum((1 for t in self.trades if t['Type'] == 'BUY'))}")
        print(f"  Sell trades: {sum((1 for t in self.trades if t['Type'] == 'SELL'))}")
        print(f"  Final portfolio value: ${results.iloc[-1]['Portfolio_Value']:,.2f}")
        return results

    def add_buy_hold_benchmark(
        self, results: pd.DataFrame, price_df: pd.DataFrame, ticker_name: str
    ) -> pd.DataFrame:
        """
        Add buy-and-hold benchmark.

        Benchmark Logic:
        ----------------
        - Buy at first Open with commission
        - Hold throughout period
        - Value at Close each day (no execution, just holding)
        - Returns are Close-to-Close
        - Sell at last Close with commission

        Parameters:
        -----------
        results : pd.DataFrame
            Main results dataframe (modified in place)
        price_df : pd.DataFrame
            Price data with Date, Open, Close columns
        ticker_name : str
            Name for the benchmark (e.g., 'SPY', 'URTH')

        Returns:
        --------
        pd.DataFrame with benchmark columns added
        """
        price_aligned = price_df[["Date", "Open", "Close"]].rename(
            columns={"Open": f"{ticker_name}_Open", "Close": f"{ticker_name}_Close"}
        )
        results = results.merge(price_aligned, on="Date", how="left")
        first_open = results.loc[0, f"{ticker_name}_Open"]
        commission_buy = self.initial_capital * self.commission_rate
        available_for_shares = self.initial_capital - commission_buy
        shares = available_for_shares / first_open
        col_prefix = f"{ticker_name}_BH"
        results[f"{col_prefix}_Value"] = shares * results[f"{ticker_name}_Close"]
        results.loc[0, f"{col_prefix}_Value"] = self.initial_capital - commission_buy
        last_idx = len(results) - 1
        last_close = results.loc[last_idx, f"{ticker_name}_Close"]
        gross_proceeds = shares * last_close
        commission_sell = gross_proceeds * self.commission_rate
        results.loc[last_idx, f"{col_prefix}_Value"] = gross_proceeds - commission_sell
        results[f"{col_prefix}_Returns"] = results[f"{ticker_name}_Close"].pct_change()
        results[f"{col_prefix}_Cumulative"] = (1 + results[f"{col_prefix}_Returns"]).cumprod()
        print(f"✓ {ticker_name} buy-and-hold benchmark added")
        print(f"  Shares purchased: {shares:.2f}")
        print(f"  Buy commission: ${commission_buy:,.2f}")
        print(f"  Sell commission: ${commission_sell:,.2f}")
        print(f"  Final value: ${results.loc[last_idx, f'{col_prefix}_Value']:,.2f}")
        return results


class PerformanceMetrics:
    """Calculate and display comprehensive performance metrics."""

    @staticmethod
    def calculate_all_metrics(results: pd.DataFrame, rf_rate: float = 0.02) -> Dict:
        """
        Calculate performance metrics for strategy and benchmarks.

        Metrics Calculated:
        -------------------
        - Total Return: (Final Value / Initial Value) - 1
        - CAGR: Compound Annual Growth Rate
        - Volatility: Annualized standard deviation
        - Sharpe Ratio: Risk-adjusted return
        - Max Drawdown: Largest peak-to-trough decline
        - Win Rate: Percentage of positive return days
        - Final Value: Ending portfolio value

        Parameters:
        -----------
        results : pd.DataFrame
            Backtest results
        rf_rate : float
            Risk-free rate (annual)

        Returns:
        --------
        Dict with metrics for strategy and benchmarks
        """
        metrics = {}
        strategy_returns = results["Strategy_Returns"].dropna()
        n_years = len(strategy_returns) / 252
        metrics["Strategy"] = {
            "Total Return": results.iloc[-1]["Portfolio_Value"] / results.iloc[0]["Portfolio_Value"]
            - 1,
            "CAGR": (
                (results.iloc[-1]["Portfolio_Value"] / results.iloc[0]["Portfolio_Value"])
                ** (1 / n_years)
                - 1
                if n_years > 0
                else 0
            ),
            "Volatility": strategy_returns.std() * np.sqrt(252),
            "Sharpe": (
                np.sqrt(252) * (strategy_returns.mean() - rf_rate / 252) / strategy_returns.std()
                if strategy_returns.std() > 0
                else 0
            ),
            "Max Drawdown": (
                (strategy_returns + 1).cumprod()
                / (strategy_returns + 1).cumprod().expanding().max()
                - 1
            ).min(),
            "Win Rate": (strategy_returns > 0).sum() / len(strategy_returns),
            "Avg Daily Return": strategy_returns.mean(),
            "Final Value": results.iloc[-1]["Portfolio_Value"],
        }
        for benchmark in ["SPY", "URTH"]:
            col_prefix = f"{benchmark}_BH"
            if f"{col_prefix}_Returns" in results.columns:
                bench_returns = results[f"{col_prefix}_Returns"].dropna()
                metrics[benchmark] = {
                    "Total Return": results.iloc[-1][f"{col_prefix}_Value"]
                    / results.iloc[0][f"{col_prefix}_Value"]
                    - 1,
                    "CAGR": (
                        (
                            results.iloc[-1][f"{col_prefix}_Value"]
                            / results.iloc[0][f"{col_prefix}_Value"]
                        )
                        ** (1 / n_years)
                        - 1
                        if n_years > 0
                        else 0
                    ),
                    "Volatility": bench_returns.std() * np.sqrt(252),
                    "Sharpe": (
                        np.sqrt(252) * (bench_returns.mean() - rf_rate / 252) / bench_returns.std()
                        if bench_returns.std() > 0
                        else 0
                    ),
                    "Max Drawdown": (
                        (bench_returns + 1).cumprod()
                        / (bench_returns + 1).cumprod().expanding().max()
                        - 1
                    ).min(),
                    "Avg Daily Return": bench_returns.mean(),
                    "Final Value": results.iloc[-1][f"{col_prefix}_Value"],
                }
        return metrics

    @staticmethod
    def print_metrics(metrics: Dict):
        """Print formatted performance metrics table."""
        print("\n" + "=" * 100)
        print("PERFORMANCE METRICS SUMMARY")
        print("=" * 100)
        for name, m in metrics.items():
            print(f"\n{name}:")
            print(f"  Total Return:        {m['Total Return']:>10.2%}")
            print(f"  CAGR:                {m['CAGR']:>10.2%}")
            print(f"  Volatility:          {m['Volatility']:>10.2%}")
            print(f"  Sharpe Ratio:        {m['Sharpe']:>10.2f}")
            print(f"  Max Drawdown:        {m['Max Drawdown']:>10.2%}")
            print(f"  Avg Daily Return:    {m['Avg Daily Return']:>10.4%}")
            if "Win Rate" in m:
                print(f"  Win Rate:            {m['Win Rate']:>10.2%}")
            print(f"  Final Value:         ${m['Final Value']:>10,.2f}")
        if "Strategy" in metrics:
            print(f"\n{'=' * 100}")
            print("RELATIVE PERFORMANCE")
            print("=" * 100)
            if "SPY" in metrics:
                print(f"\nStrategy vs SPY:")
                print(
                    f"  Excess Return:       {metrics['Strategy']['Total Return'] - metrics['SPY']['Total Return']:>10.2%}"
                )
                print(
                    f"  Excess CAGR:         {metrics['Strategy']['CAGR'] - metrics['SPY']['CAGR']:>10.2%}"
                )
                alpha_spy = metrics["Strategy"]["CAGR"] - metrics["SPY"]["CAGR"]
                print(f"  Alpha:               {alpha_spy:>10.2%}")
            if "URTH" in metrics:
                print(f"\nStrategy vs URTH (Global Benchmark):")
                print(
                    f"  Excess Return:       {metrics['Strategy']['Total Return'] - metrics['URTH']['Total Return']:>10.2%}"
                )
                print(
                    f"  Excess CAGR:         {metrics['Strategy']['CAGR'] - metrics['URTH']['CAGR']:>10.2%}"
                )
                alpha_urth = metrics["Strategy"]["CAGR"] - metrics["URTH"]["CAGR"]
                print(f"  Alpha:               {alpha_urth:>10.2%}")
        print("=" * 100 + "\n")


def plot_results(results: pd.DataFrame, config: StrategyConfig):
    """
    Create comprehensive visualization plots.

    Plots Created:
    --------------
    1. Price with regime overlay (scatter plot)
    2. Cumulative returns comparison
    3. Strategy drawdown
    4. Feature distributions by regime
    """
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(4, 2, hspace=0.3, wspace=0.3)
    ax1 = fig.add_subplot(gs[0, :])
    regime_colors = {"Bearish": "red", "Neutral": "gray", "Bullish": "green"}
    for regime, color in regime_colors.items():
        mask = results["Regime"] == regime
        if mask.sum() > 0:
            ax1.scatter(
                results.loc[mask, "Date"],
                results.loc[mask, "Close"],
                c=color,
                label=regime,
                alpha=0.6,
                s=15,
            )
    ax1.set_ylabel("Price ($)", fontsize=11)
    ax1.set_title(
        f"{config.MAIN_TICKER} Price with HMM Regimes (Walk-Forward Validated)",
        fontsize=13,
        fontweight="bold",
    )
    ax1.legend(loc="best")
    ax1.grid(alpha=0.3)
    ax2 = fig.add_subplot(gs[1, :])
    ax2.plot(
        results["Date"],
        100 * results["Cumulative_Returns"],
        label="HMM Strategy",
        linewidth=2,
        color="blue",
    )
    if "SPY_BH_Cumulative" in results.columns:
        ax2.plot(
            results["Date"],
            100 * results["SPY_BH_Cumulative"],
            label="SPY Buy & Hold",
            linewidth=2,
            alpha=0.7,
            color="orange",
        )
    if "URTH_BH_Cumulative" in results.columns:
        ax2.plot(
            results["Date"],
            100 * results["URTH_BH_Cumulative"],
            label="URTH Buy & Hold",
            linewidth=2,
            alpha=0.7,
            color="green",
        )
    ax2.set_ylabel("Cumulative Returns (Base 100)", fontsize=11)
    ax2.set_title("Strategy vs Benchmarks Performance", fontsize=13, fontweight="bold")
    ax2.legend(loc="best")
    ax2.grid(alpha=0.3)
    ax3 = fig.add_subplot(gs[2, :])
    strategy_cum = (1 + results["Strategy_Returns"]).cumprod()
    strategy_dd = (strategy_cum / strategy_cum.expanding().max() - 1) * 100
    ax3.fill_between(results["Date"], strategy_dd, 0, alpha=0.3, color="red")
    ax3.plot(results["Date"], strategy_dd, color="darkred", linewidth=1.5)
    ax3.set_xlabel("Date", fontsize=11)
    ax3.set_ylabel("Drawdown (%)", fontsize=11)
    ax3.set_title("Strategy Drawdown", fontsize=13, fontweight="bold")
    ax3.grid(alpha=0.3)
    ax4 = fig.add_subplot(gs[3, 0])
    for regime, color in regime_colors.items():
        mask = results["Regime"] == regime
        if mask.sum() > 0:
            data = results.loc[mask, "ROC63"].dropna()
            if len(data) > 0:
                ax4.hist(data, bins=30, alpha=0.5, label=regime, color=color)
    ax4.set_xlabel("ROC63", fontsize=10)
    ax4.set_ylabel("Frequency", fontsize=10)
    ax4.set_title("ROC63 Distribution by Regime", fontsize=12, fontweight="bold")
    ax4.legend()
    ax4.grid(alpha=0.3)
    ax5 = fig.add_subplot(gs[3, 1])
    for regime, color in regime_colors.items():
        mask = results["Regime"] == regime
        if mask.sum() > 0:
            data = results.loc[mask, "YZ_Vol_Pct"].dropna()
            if len(data) > 0:
                ax5.hist(data, bins=30, alpha=0.5, label=regime, color=color)
    ax5.set_xlabel("YZ Volatility Percentile", fontsize=10)
    ax5.set_ylabel("Frequency", fontsize=10)
    ax5.set_title("YZ Vol Percentile Distribution by Regime", fontsize=12, fontweight="bold")
    ax5.legend()
    ax5.grid(alpha=0.3)
    plt.suptitle("HMM Regime Strategy - Complete Analysis", fontsize=15, fontweight="bold", y=0.995)
    plt.show()


def create_regime_analysis(results: pd.DataFrame) -> pd.DataFrame:
    """
    Create detailed regime analysis showing forward returns by regime.

    This validates that the regime detection is working correctly:
    - Bullish regime should have highest forward returns
    - Bearish regime should have lowest forward returns

    Parameters:
    -----------
    results : pd.DataFrame
        Backtest results with Regime column

    Returns:
    --------
    pd.DataFrame with regime statistics
    """
    results_copy = results.copy()
    results_copy["Forward_Return_OO"] = results_copy["Returns_OO"].shift(-1)
    results_copy["Forward_Return_CC"] = results_copy["Returns_CC"].shift(-1)
    regime_stats = []
    for regime in ["Bearish", "Neutral", "Bullish"]:
        mask = results_copy["Regime"] == regime
        if mask.sum() > 0:
            fwd_oo = results_copy.loc[mask, "Forward_Return_OO"].dropna()
            fwd_cc = results_copy.loc[mask, "Forward_Return_CC"].dropna()
            regime_stats.append(
                {
                    "Regime": regime,
                    "Count": mask.sum(),
                    "Avg Forward Return (OO)": fwd_oo.mean(),
                    "Avg Forward Return (CC)": fwd_cc.mean(),
                    "Win Rate (OO)": (fwd_oo > 0).sum() / len(fwd_oo) if len(fwd_oo) > 0 else 0,
                    "Win Rate (CC)": (fwd_cc > 0).sum() / len(fwd_cc) if len(fwd_cc) > 0 else 0,
                }
            )
    df_stats = pd.DataFrame(regime_stats)
    print("\n" + "=" * 100)
    print("REGIME VALIDATION - FORWARD RETURNS BY REGIME")
    print("=" * 100)
    print("\nThis table shows whether regimes predict future returns correctly:")
    print("Expected: Bullish > Neutral > Bearish\n")
    print(df_stats.to_string(index=False))
    print("=" * 100 + "\n")
    return df_stats


def create_trade_log_sample(results: pd.DataFrame, n_samples: int = 15) -> pd.DataFrame:
    """
    Create sample trade log for validation.

    Shows actual trades with context to verify signal-execution alignment.
    """
    trade_rows = results[results["Trades"].isin(["BUY", "SELL"])].copy()
    if len(trade_rows) == 0:
        print("No trades executed")
        return pd.DataFrame()
    sample_size = min(n_samples, len(trade_rows))
    sample_indices = np.random.choice(trade_rows.index, sample_size, replace=False)
    sample = results.loc[sorted(sample_indices)].copy()
    log_cols = [
        "Date",
        "Regime",
        "Signal",
        "Position",
        "Trades",
        "Open",
        "Strategy_Returns",
        "Portfolio_Value",
    ]
    available_cols = [col for col in log_cols if col in sample.columns]
    log = sample[available_cols].copy()
    return log


def main():
    """
    Main execution function with complete workflow.

    Workflow:
    ---------
    1. Configure strategy parameters
    2. Fetch data from Yahoo Finance
    3. Engineer features (ROC63, Yang-Zhang Vol, Percentiles)
    4. Perform walk-forward HMM regime detection
    5. Run backtest with realistic execution
    6. Add benchmarks for comparison
    7. Calculate performance metrics
    8. Create visualizations
    9. Export results
    """
    print("\n" + "=" * 100)
    print("HMM REGIME STRATEGY - ROC63 & YANG-ZHANG VOLATILITY")
    print("=" * 100)
    config = StrategyConfig()
    print(f"\nStrategy Configuration:")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Ticker: {config.MAIN_TICKER}")
    print(f"  Benchmark: {config.BENCHMARK_TICKER}")
    print(f"  Initial Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Commission Rate: {config.COMMISSION_RATE:.3%}")
    print(f"\nFeature Parameters:")
    print(f"  ROC Period: {config.ROC_PERIOD} days")
    print(f"  Yang-Zhang Window: {config.YANG_ZHANG_WINDOW} days")
    print(f"  Percentile Window: {config.PERCENTILE_WINDOW} days")
    print(f"\nWalk-Forward Parameters:")
    print(f"  Min Training Days: {config.MIN_TRAINING_DAYS}")
    print(f"  Refit Frequency: {config.REFIT_FREQUENCY} days")
    print("\n" + "-" * 100)
    print("STEP 1: DATA ACQUISITION")
    print("-" * 100)
    fetcher = DataFetcher()
    df_spy = fetcher.fetch_historical_data(config.MAIN_TICKER, config.START_DATE, config.END_DATE)
    df_urth = fetcher.fetch_historical_data(
        config.BENCHMARK_TICKER, config.START_DATE, config.END_DATE
    )
    print("\n" + "-" * 100)
    print("STEP 2: FEATURE ENGINEERING")
    print("-" * 100)
    df_features = FeatureEngine.prepare_features(df_spy, config)
    print("\n" + "-" * 100)
    print("STEP 3: WALK-FORWARD REGIME DETECTION")
    print("-" * 100)
    detector = WalkForwardHMMRegimeDetector(config)
    df_regimes, folds = detector.walk_forward_predict(df_features)
    print("\n" + "-" * 100)
    print("STEP 4: BACKTESTING")
    print("-" * 100)
    engine = BacktestEngine(config.INITIAL_CAPITAL, config.COMMISSION_RATE)
    results = engine.run_backtest(df_regimes)
    print("\nAdding benchmarks...")
    results = engine.add_buy_hold_benchmark(results, df_regimes, "SPY")
    results = engine.add_buy_hold_benchmark(results, df_urth, "URTH")
    print("\n" + "-" * 100)
    print("STEP 5: PERFORMANCE ANALYSIS")
    print("-" * 100)
    metrics = PerformanceMetrics.calculate_all_metrics(results)
    PerformanceMetrics.print_metrics(metrics)
    print("\n" + "-" * 100)
    print("STEP 6: VALIDATION & DIAGNOSTICS")
    print("-" * 100)
    regime_stats = create_regime_analysis(results)
    print("\n" + "-" * 100)
    print("SAMPLE TRADE LOG (Verifying Signal-Execution Alignment)")
    print("-" * 100)
    trade_log = create_trade_log_sample(results, n_samples=15)
    if len(trade_log) > 0:
        print(trade_log.to_string(index=False))
    print("\n" + "=" * 100)
    print("SUMMARY STATISTICS")
    print("=" * 100)
    print(f"  Total trading days: {len(results)}")
    print(f"  Days with predictions: {results['Regime'].notna().sum()}")
    print(f"  Days with Signal=1 (Long): {(results['Signal'] == 1).sum()}")
    print(f"  Days with Position=1 (Holding): {(results['Position'] == 1).sum()}")
    print(f"  Total trades executed: {len(engine.trades)}")
    regime_counts = results["Regime"].value_counts()
    print(f"\nRegime Distribution:")
    for regime in ["Bullish", "Neutral", "Bearish"]:
        count = regime_counts.get(regime, 0)
        pct = count / len(results) * 100 if len(results) > 0 else 0
        print(f"  {regime:10s}: {count:4d} days ({pct:5.1f}%)")
    print("\n" + "-" * 100)
    print("STEP 7: GENERATING VISUALIZATIONS")
    print("-" * 100)
    plot_results(results, config)
    print("\n" + "-" * 100)
    print("STEP 8: EXPORTING RESULTS")
    print("-" * 100)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_file = f"hmm_roc_yangzhang_results_{timestamp}.csv"
    results.to_csv(results_file, index=False)
    print(f"✓ Main results exported: {results_file}")
    trades_file = f"hmm_roc_yangzhang_trades_{timestamp}.csv"
    pd.DataFrame(engine.trades).to_csv(trades_file, index=False)
    print(f"✓ Trade log exported: {trades_file}")
    folds_file = f"hmm_roc_yangzhang_folds_{timestamp}.csv"
    pd.DataFrame(folds).to_csv(folds_file, index=False)
    print(f"✓ Fold information exported: {folds_file}")
    regime_file = f"hmm_roc_yangzhang_regime_stats_{timestamp}.csv"
    regime_stats.to_csv(regime_file, index=False)
    print(f"✓ Regime statistics exported: {regime_file}")
    print("\n" + "=" * 100)
    print("EXECUTION COMPLETE")
    print("=" * 100)
    print("\nKey Results:")
    print(f"  Strategy Total Return: {metrics['Strategy']['Total Return']:.2%}")
    print(f"  Strategy CAGR: {metrics['Strategy']['CAGR']:.2%}")
    print(f"  Strategy Sharpe: {metrics['Strategy']['Sharpe']:.2f}")
    print(f"  Strategy Max Drawdown: {metrics['Strategy']['Max Drawdown']:.2%}")
    if "URTH" in metrics:
        print(f"\n  URTH Total Return: {metrics['URTH']['Total Return']:.2%}")
        print(
            f"  Excess Return vs URTH: {metrics['Strategy']['Total Return'] - metrics['URTH']['Total Return']:.2%}"
        )
    print("\nFiles exported with timestamp:", timestamp)
    print("=" * 100 + "\n")
    return (results, metrics, folds, regime_stats)


if __name__ == "__main__":
    results, metrics, folds, regime_stats = main()
